# Fine-tune Qwen2.5-1.5B to turn customer messages into structured tickets (EN + AR)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Total time is about 20–25 minutes on the free tier.

Steps: install → measure the base model → fine-tune with LoRA (Unsloth) → measure again on the same hand-written test set → save results.

In [ ]:
!git clone https://github.com/KirollosMessak/ai-ticket-extractor.git
%cd ai-ticket-extractor
!pip install -q unsloth

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0))

## 1. Baseline: the base model *before* fine-tuning

Same prompt, same 60 hand-written test messages that are never used for training.

In [ ]:
!python eval_model.py --model unsloth/Qwen2.5-1.5B-Instruct --name base

## 2. Load the model and add LoRA adapters

- 1.5B parameters fits in 16-bit on a T4, so 4-bit quantization isn't needed (slightly better quality).
- LoRA is added to the attention **and** MLP layers.
- `r=16`, `lora_alpha=16` is a standard starting point.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=1024,
    load_in_4bit=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. Prepare the data

Each example becomes a chat: system prompt → customer message → JSON answer. The prompt is the same one used for evaluation (`common.py`).

In [ ]:
from datasets import load_dataset
from common import build_messages

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(build_messages(ex["text"], ex["label"]), tokenize=False)}

data = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})
data = data.map(to_text, remove_columns=["label"])
print(data)
print(data["train"][0]["text"])

## 4. Train

- 3 epochs over 900 examples (enough to learn the format without memorizing).
- `train_on_responses_only`: the loss is computed only on the JSON answer, not on the prompt.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data["train"],
    eval_dataset=data["val"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        logging_steps=10,
        eval_strategy="epoch",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        output_dir="outputs/checkpoints",
        save_strategy="no",
        report_to="none",
        seed=3407,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

## 5. Quick try

In [ ]:
from eval_model import predict

FastLanguageModel.for_inference(model)
for msg in [
    "الأوردر رقم ١٢٣٤٥ لسه موصلش وأنا محتاجه بكرة ضروري",
    "Hi! Can I swap my shoes for a bigger size? order 8812",
    "el visa msh rady tedfa3, 7aga tezah2",
]:
    print(msg, "->", predict(model, tokenizer, [msg])[0])

## 6. Save the model

- `outputs/lora`: just the LoRA adapter (small)
- `outputs/merged`: the full model with the adapter merged in (used for evaluation and the demo)

In [ ]:
model.save_pretrained("outputs/lora")
tokenizer.save_pretrained("outputs/lora")
model.save_pretrained_merged("outputs/merged", tokenizer, save_method="merged_16bit")

# Free GPU memory before evaluating in a separate process
import gc
del trainer, model
gc.collect()
torch.cuda.empty_cache()

## 7. Evaluate the fine-tuned model on the same test set

In [ ]:
!python eval_model.py --model outputs/merged --name finetuned

## 8. Before vs. after

In [ ]:
import json
base = json.load(open("results/base.json"))
ft = json.load(open("results/finetuned.json"))
print(f"{'metric':<16}{'base':>8}{'fine-tuned':>12}")
for k in base["metrics"]:
    print(f"{k:<16}{base['metrics'][k]:>8.0%}{ft['metrics'][k]:>12.0%}")
for lang in ft["exact_match_by_language"]:
    print(f"{'exact (' + lang + ')':<16}{base['exact_match_by_language'][lang]:>8.0%}{ft['exact_match_by_language'][lang]:>12.0%}")

## 9. Download the results

Send `results.zip` (it contains `base.json` and `finetuned.json`) so the README numbers come from real outputs.

In [ ]:
!zip -r results.zip results
from google.colab import files
files.download("results.zip")

## 10. (Optional) Publish the model on Hugging Face

Create a **write** token at https://huggingface.co/settings/tokens, then run this cell.

In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()
REPO_ID = "YOUR_HF_USERNAME/ticket-extractor-qwen1.5b"  # ← change this
HfApi().create_repo(REPO_ID, exist_ok=True)
HfApi().upload_folder(folder_path="outputs/merged", repo_id=REPO_ID)
print("https://huggingface.co/" + REPO_ID)